# Sannidhi · make all audio (no install)

Runs in your browser on Google Colab. You need your **Sarvam API key** and a zip of the repo folders `content/`, `narration/` and `tools/` (zip them together as `sannidhi-content.zip`).

Press **Runtime → Run all**. At the end a zip of the new audio downloads; unzip it at the repo root.

- Story narration (te, hi, en) → `content/audio/stories/…` (for release, after you listen and approve)
- Draft aṣṭottaram and mūla mantra → `content/audio/…` (**testing only**; release uses a human recording)

In [ ]:
#@title 1 · Upload sannidhi-content.zip
from google.colab import files
import zipfile, os
up = files.upload()
name = next(iter(up))
os.makedirs('repo', exist_ok=True)
zipfile.ZipFile(name).extractall('repo')
print(sorted(os.listdir('repo')))

In [ ]:
#@title 2 · Sarvam key and tools
import getpass, os, subprocess
os.environ['SARVAM_API_KEY'] = getpass.getpass('Sarvam API key: ')
subprocess.run('apt-get -qq install -y ffmpeg > /dev/null', shell=True)
print('ready')

In [ ]:
#@title 3 · Check the story lines (no cost)
!cd repo && python tools/make_narration.py --dry-run | head -60

In [ ]:
#@title 4 · Audition voices (beat 1 of each story, every shortlist voice) — optional
AUDITION = False #@param {type:'boolean'}
if AUDITION:
    !cd repo && python tools/make_narration.py --audition
    from IPython.display import Audio, display
    import glob
    for f in sorted(glob.glob('repo/narration/auditions/*.m4a'))[:40]:
        print(f); display(Audio(f))

In [ ]:
#@title 5 · Make story narration (te, hi, en)
!cd repo && python tools/make_narration.py

In [ ]:
#@title 6 · Make DRAFT aṣṭottaram and mūla mantra (testing only)
# The aṣṭottaram names must be imported first (tools/import_ashtottara.py); otherwise this step says so and skips.
!cd repo && python tools/make_sacred_draft_audio.py --text ganesha.ashtottara || true
!cd repo && python tools/make_sacred_draft_audio.py --mantra ganesha.mantra.mula --as-text ganesha.mula-mantra

In [ ]:
#@title 7 · Listen to a few
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob('repo/content/audio/**/*.m4a', recursive=True))[:12]:
    print(f); display(Audio(f))

In [ ]:
#@title 8 · Download the audio
import shutil
shutil.make_archive('sannidhi-audio', 'zip', 'repo', 'content/audio')
files.download('sannidhi-audio.zip')